# SQL Server Smoke Test — VDI

**What this does:** confirms this VDI can connect to `RSD-RBSQLDEV` / `BI_SANDBOX` from Python.

**How to use it:** run each cell one at a time using **Shift + Enter**. Don't skip cells. Don't edit code.

Each cell tells you ✅ if it passed, or ❌ with exactly what to do next if it didn't.

There's only **one** action you need to take manually — after Cell 1, do **Kernel → Restart** from the menu. The notebook reminds you.


## Cell 1 — Install the two packages we need

Uses `%pip` which installs into the *correct* Python for this kernel. Safe to re-run.

In [ ]:
%pip install --quiet pyodbc pandas
print('\n' + '─' * 60)
print('✅ Install complete.')
print('─' * 60)
print('\n👉 NEXT: click Kernel → Restart in the top menu.')
print('   Then come back here and run Cell 2.')
print('   (If you skip the restart, Cell 2 will fail with')
print('    "No module named pyodbc".)')

## Cell 2 — Verify imports work

If you forgot to restart the kernel after Cell 1, this cell tells you.

In [ ]:
import sys

missing = []
for mod in ['pyodbc', 'pandas']:
    try:
        __import__(mod)
    except ImportError:
        missing.append(mod)

print('─' * 60)
if missing:
    print(f'❌ Cannot import: {missing}')
    print()
    print('FIX: go to the top menu → Kernel → Restart.')
    print('     Then run Cell 2 again (skip Cell 1, it already ran).')
    print()
    print('If you already restarted and still see this, screenshot')
    print('this cell and send to Prosper.')
else:
    import pyodbc, pandas as pd
    print(f'✅ pyodbc {pyodbc.version}   pandas {pd.__version__}')
    print(f'   Python: {sys.version.split()[0]}')
    print()
    print('👉 NEXT: run Cell 3.')
print('─' * 60)

## Cell 3 — Check the SQL Server ODBC driver is installed on this VDI

In [ ]:
import pyodbc

drivers = [d for d in pyodbc.drivers() if 'SQL Server' in d]

print('─' * 60)
if not drivers:
    SQL_DRIVER = None
    print('❌ No SQL Server ODBC driver found on this VDI.')
    print()
    print('All drivers Python can see:')
    for d in pyodbc.drivers():
        print(f'    {d}')
    print()
    print('FIX: stop here. Screenshot this cell and send to Prosper.')
    print('     IT will need to install "ODBC Driver 18 for SQL Server".')
else:
    # Prefer Driver 18, else 17, else whatever's there
    SQL_DRIVER = next((d for d in drivers if '18' in d),
                     next((d for d in drivers if '17' in d), drivers[-1]))
    print(f'✅ Found {len(drivers)} SQL Server driver(s):')
    for d in drivers:
        mark = '◀ will use this' if d == SQL_DRIVER else ''
        print(f'    {d}  {mark}')
    print()
    print('👉 NEXT: run Cell 4.')
print('─' * 60)

## Cell 4 — Can we even reach the server on the network?

Tests the TCP connection to port 1433 before involving SQL or credentials. If this fails, the problem is networking, not SQL.

In [ ]:
import socket

SQL_SERVER = 'RSD-RBSQLDEV'
SQL_PORT   = 1433

print('─' * 60)

# DNS
try:
    resolved_ip = socket.gethostbyname(SQL_SERVER)
    print(f'✅ DNS: {SQL_SERVER} → {resolved_ip}')
except socket.gaierror as e:
    resolved_ip = None
    print(f'❌ DNS: cannot resolve {SQL_SERVER}  ({e})')
    print()
    print('FIX: the VDI cannot see the internal DNS for the server.')
    print('     Stop here and send Prosper a screenshot of this cell.')

# TCP port
if resolved_ip:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(5)
    rc = sock.connect_ex((resolved_ip, SQL_PORT))
    sock.close()

    if rc == 0:
        print(f'✅ TCP port {SQL_PORT} is open on {resolved_ip}')
        print()
        print('👉 NEXT: run Cell 5.')
    else:
        print(f'❌ TCP port {SQL_PORT} is NOT reachable (error code {rc})')
        print()
        print('FIX: server is unreachable from this VDI.')
        print('     - Are you sure you ran this on the VDI (not your Mac)?')
        print('     - Stop here and send Prosper a screenshot.')
print('─' * 60)

## Cell 5 — Connect to SQL Server using your Windows login

No password — `Trusted_Connection=yes` uses your VDI identity automatically.

In [ ]:
import pyodbc

SQL_DATABASE = 'BI_SANDBOX'

conn_str = (
    f'DRIVER={{{SQL_DRIVER}}};'
    f'SERVER={SQL_SERVER},{SQL_PORT};'
    f'DATABASE={SQL_DATABASE};'
    'Encrypt=yes;'
    'TrustServerCertificate=yes;'
    'Trusted_Connection=yes;'
)

print('─' * 60)
print(f'Connecting to {SQL_SERVER}:{SQL_PORT}/{SQL_DATABASE} as your Windows account...')

conn = None
try:
    conn = pyodbc.connect(conn_str, timeout=15)
    cur = conn.cursor()
    cur.execute('SELECT SYSTEM_USER, SUSER_SNAME(), DB_NAME(), @@SERVERNAME')
    user, login, db, server = cur.fetchone()
    print()
    print(f'✅ CONNECTED')
    print(f'   Server       : {server}')
    print(f'   Database     : {db}')
    print(f'   Logged in as : {user}')
    print(f'   SUSER_SNAME  : {login}')
    print()
    print('👉 NEXT: run Cell 6.')
except pyodbc.Error as e:
    err = str(e)
    print(f'❌ Connection failed.')
    print()
    print('Driver error:')
    print(f'   {err[:400]}')
    print()
    if 'Login failed' in err or '18456' in err:
        print('FIX: SQL Server rejected your Windows login.')
        print('     Your AD account probably needs db_datareader on BI_SANDBOX.')
        print('     Send Prosper this screenshot — DBA will need to grant access.')
    elif 'timeout' in err.lower() or 'time-out' in err.lower():
        print('FIX: server didn\'t respond. Network issue.')
        print('     Send Prosper this screenshot.')
    else:
        print('FIX: unrecognised error. Send Prosper this full screenshot.')
print('─' * 60)

## Cell 6 — List tables you can see + read a small preview

Confirms you can actually query data, not just connect.

In [ ]:
import pandas as pd

print('─' * 60)

if conn is None:
    print('❌ Skipped — Cell 5 failed to connect. Fix Cell 5 first.')
else:
    # List tables
    tables = pd.read_sql("""
        SELECT TABLE_SCHEMA, TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME DESC
    """, conn)

    print(f'✅ Tables visible to your login in {SQL_DATABASE}: {len(tables)}')
    print()
    if len(tables) == 0:
        print('No tables visible. Connection worked but no read access yet.')
        print('Send Prosper this screenshot.')
    else:
        print('First 10 tables (most recent first):')
        print(tables.head(10).to_string(index=False))
        print()

        # Find a BASE table to preview
        base_tables = tables[tables['TABLE_NAME'].str.startswith('BASE')]
        if not base_tables.empty:
            target = base_tables.iloc[0]
            qualified = f"[{target['TABLE_SCHEMA']}].[{target['TABLE_NAME']}]"
            count_df = pd.read_sql(f'SELECT COUNT(*) AS n FROM {qualified}', conn)
            row_count = int(count_df['n'].iloc[0])
            print(f'Preview from {qualified}  ({row_count:,} rows total):')
            preview = pd.read_sql(f'SELECT TOP 5 * FROM {qualified}', conn)
            print(preview.to_string(index=False, max_cols=8))
        else:
            print('(no BASE* table to preview — only checking other tables)')

print('─' * 60)

## Cell 7 — Final result

In [ ]:
print('=' * 60)
if conn is not None:
    try:
        cur = conn.cursor()
        cur.execute('SELECT 1')
        cur.fetchone()
        print('✅ ✅ ✅  ALL CHECKS PASSED')
        print()
        print('Next steps:')
        print('  1. Screenshot Cells 5 and 6 (connection + table preview).')
        print('  2. Send the screenshot to Prosper on Teams.')
        print('  3. We move on to the real pipeline notebook.')
    except Exception as e:
        print('⚠ Connection was opened but is no longer usable.')
        print(f'   {e}')
        print('   Re-run from Cell 5.')
    finally:
        try:
            conn.close()
            print('\n(Connection closed cleanly.)')
        except Exception:
            pass
else:
    print('❌ One or more checks failed. Scroll up and find the cell with ❌.')
    print('   The fix instructions are inside that cell\'s output.')
print('=' * 60)